# Config

In [61]:
MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0
DATA_HEADER = [
    "th1",
    "th2",
    "th3",
    "in1",
    "in2",
    "in3",
    "mi1",
    "mi2",
    "mi3",
    "ri1",
    "ri2",
    "ri3",
    "pi1",
    "pi2",
    "pi3",
    "pnx",
    "pny",
    "pnz",
    # "thumb_index_tip_dist",
    # "thumb_middle_tip_dist",
    "hand_side",
]


LABEL_HEADER = [
    # finger labels
    # 1 = open, 0 = folded
    "th_open",
    "in_open",
    "mi_open",
    "ri_open",
    "pi_open",
    "ti_touch",
    "tm_touch",
]

# Load mediapipe Hand Landmarks and Opencv2

In [62]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

# Mediapipe Utils

In [63]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

# Recognition Test

In [64]:
if REC_TEST:
    detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
    hand_connections = get_drawing_utils()

    try:
        while True:
            ret, frame = capture_frame(cap)
            if not ret:
                print("프레임을 읽지 못했습니다. 종료합니다.")
                break

            rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
            annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

            # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
            display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
            cv2.imshow("MediaPipe Hand Landmarks", display_image)

            # ESC 또는 q 입력 시 종료
            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord("q"):
                break
    finally:
        cap.release()
        cv2.waitKey(-1)
        cv2.destroyAllWindows()

# Create Data

In [65]:
def parse_to_angle(detection_result):

    left_result = None
    right_result = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        v_p1 = joint[[0, 0], :]
        v_p2 = joint[[5, 17], :]
        v_palm = v_p2 - v_p1

        palm_normal = np.cross(v_palm[0], v_palm[1])
        palm_normal = palm_normal / np.linalg.norm(palm_normal)
        palm_normal = palm_normal * 500

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        tdi = np.linalg.norm(joint[4] - joint[8]) / palm_size
        tdm = np.linalg.norm(joint[4] - joint[12]) / palm_size

        full_data = np.append(angle, palm_normal)
        # full_data = np.append(full_data, tdi)
        # full_data = np.append(full_data, tdm)
        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)

        if hand_label == "Right":
            right_result = data
        elif hand_label == "Left":
            left_result = data

    return {
        "left": left_result,
        "right": right_result
    }

In [66]:
import csv
from os import path


detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()
state = {}
store = False
for head in LABEL_HEADER:
    state[head] = False


try:
    is_new_left = not path.exists("train_data_left.csv")
    is_new_right = not path.exists("train_data_right.csv")
    with open("train_data_left.csv", "a", newline="") as f_left, open(
        "train_data_right.csv", "a", newline=""
    ) as f_right:
        writer_left = csv.writer(f_left)
        writer_right = csv.writer(f_right)
        if is_new_left:
            writer_left.writerow(DATA_HEADER + LABEL_HEADER)
        if is_new_right:
            writer_right.writerow(DATA_HEADER + LABEL_HEADER)
        while True:
            key = cv2.waitKey(1) & 0xFF
            if ord("1") <= key <= ord("9"):
                state_key = LABEL_HEADER[key - ord("1")]
                state[state_key] = False if state[state_key] else True

            if ord("l") == key:
                store = False if store else True

            ret, frame = capture_frame(cap)

            if not ret:
                print("프레임을 읽지 못했습니다. 종료합니다.")
                break

            rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
            annotated_image = draw_landmarks_on_image(
                rgb_frame, detection_result, hand_connections
            )

            if store:
                angles = parse_to_angle(detection_result)

                label = [1 if v else 0 for _, v in state.items()]

                if angles["left"] is not None:
                    writer_left.writerow(angles['left'][0].tolist() + label)
                if angles["right"] is not None:
                    writer_right.writerow(angles['right'][0].tolist() + label)
                    # print(len(angles['right'][0].tolist()), len(DATA_HEADER))

            # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
            display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

            # ------------------------------
            # 1) 위쪽: 저장 상태 표시
            # ------------------------------
            store_text = f"STORE: {'ON' if store else 'OFF'}"
            store_color = (0, 255, 0) if store else (0, 0, 255)

            cv2.putText(
                display_image,
                store_text,
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                store_color,
                2,
                cv2.LINE_AA,
            )

            # ------------------------------
            # 2) 아래쪽: 현재 선택된 라벨 나열
            # ------------------------------
            active_labels = [k for k, v in state.items() if v]
            active_text = "ACTIVE: " + (
                ", ".join(active_labels) if active_labels else "None"
            )

            h, w = display_image.shape[:2]

            cv2.putText(
                display_image,
                active_text,
                (10, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (100, 200, 255),
                2,
                cv2.LINE_AA,
            )

            cv2.imshow("MediaPipe Hand Landmarks", display_image)

            if key == 27 or key == ord("q"):
                break
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()